In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from polluted import step1, step2, step3, fix1, evaluation, prompts

load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_resourceonly_seed52_ratio0.3_polluted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10
fix_repetition = 3
current_df = df.copy()  

for iteration in range(fix_repetition):
    print(f"\n{'='*40}")
    print(f">>> ITERATION {iteration + 1} / {fix_repetition}")
    print(f"{'='*40}")
    unique_activities = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(unique_activities, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP1, prompts.USER_PROMPT_POLLUTED_STEP1)
    
    if not res_s1['data']:
        print(f">>> No more polluted candidates found in iteration {iteration + 1}. Stopping loop.")
        break
    
    print(f"Candidates found ({len(res_s1['data'])} total): {res_s1['data'][:5]} ...")

    context_json = step2.get_polluted_context(current_df, set(res_s1['data']))
    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, context_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP2, prompts.USER_PROMPT_POLLUTED_STEP2)
    s2_output_json = json.dumps(res_s2["summarized_context"], indent=2, ensure_ascii=False)
    print(f"Sample Context: {res_s2['summarized_context'][0] if res_s2['summarized_context'] else 'None'}")

    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, s2_output_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP3, prompts.USER_PROMPT_POLLUTED_STEP3)
    
    if not res_s3:
        print(f">>> No clusters formed in iteration {iteration + 1}.")
        continue
    else:
        print(f"\n>>> Cluster Preview (Total: {len(res_s3)} groups)")
        for i, (clean, variants) in enumerate(res_s3.items()):
            if i >= 3: break  # 3그룹까지만 출력
            print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s3)
    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)
    print(f">>> Iteration {iteration + 1} complete.")



>>> ITERATION 1 / 3
>>> Running Step 1  with 10 repetitions...
Candidates found (235 total): ['Check for completeness', 'Check for completeness-Clerk-000001', 'Check for completeness-Clerk-000002', 'Check for completeness-Clerk-000003', 'Check for completeness-Clerk-000004'] ...
>>> Running Step 2 (Validation Retry Mode, Max: 10)
>>> Step 2 success: All 235 activities summarized.
Sample Context: {'activity': 'Check for completeness', 'predecessors': 'Request or additional information received', 'successors': 'Either request more information or perform detailed checks'}
>>> Running Step 3  with 10 repetitions...

>>> Cluster Preview (Total: 10 groups)
  - Check for completeness: ['Check for completeness-Clerk-000001', 'Check for completeness-Clerk-000002', 'Check for completeness-Clerk-000003'] ... (+27 more)
  - Deliver card: ['Deliver card-Manager-000001', 'Deliver card-Manager-000002', 'Deliver card-Manager-000003'] ... (+27 more)
  - Make decision: ['Make decision-Manager-000001', 

In [2]:
LOG_NAME = "pub_resourceonly_seed52_ratio0.3_polluted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10
fix_repetition = 3
current_df = df.copy()  

for iteration in range(fix_repetition):
    print(f"\n{'='*40}")
    print(f">>> ITERATION {iteration + 1} / {fix_repetition}")
    print(f"{'='*40}")
    unique_activities = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(unique_activities, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP1, prompts.USER_PROMPT_POLLUTED_STEP1)
    
    if not res_s1['data']:
        print(f">>> No more polluted candidates found in iteration {iteration + 1}. Stopping loop.")
        break
    
    print(f"Candidates found ({len(res_s1['data'])} total): {res_s1['data'][:5]} ...")

    context_json = step2.get_polluted_context(current_df, set(res_s1['data']))
    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, context_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP2, prompts.USER_PROMPT_POLLUTED_STEP2)
    s2_output_json = json.dumps(res_s2["summarized_context"], indent=2, ensure_ascii=False)
    print(f"Sample Context: {res_s2['summarized_context'][0] if res_s2['summarized_context'] else 'None'}")

    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, s2_output_json, 
                             prompts.SYSTEM_PROMPT_POLLUTED_STEP3, prompts.USER_PROMPT_POLLUTED_STEP3)
    
    if not res_s3:
        print(f">>> No clusters formed in iteration {iteration + 1}.")
        continue
    else:
        print(f"\n>>> Cluster Preview (Total: {len(res_s3)} groups)")
        for i, (clean, variants) in enumerate(res_s3.items()):
            if i >= 3: break  # 3그룹까지만 출력
            print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s3)
    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)
    print(f">>> Iteration {iteration + 1} complete.")



>>> ITERATION 1 / 3
>>> Running Step 1  with 10 repetitions...
Candidates found (200 total): ['Bring drinks', 'Bring drinks-Barman-000001', 'Bring drinks-Barman-000002', 'Bring drinks-Barman-000003', 'Bring drinks.Barman-000001'] ...
>>> Running Step 2 (Validation Retry Mode, Max: 10)
>>> Step 2 success: All 200 activities summarized.
Sample Context: {'activity': 'Bring drinks', 'predecessors': 'Drink and meal preparation completion', 'successors': 'Drink delivery review and additional serving'}
>>> Running Step 3  with 10 repetitions...

>>> Cluster Preview (Total: 10 groups)
  - Bring drinks: ['Bring drinks-Barman-000001', 'Bring drinks-Barman-000002', 'Bring drinks-Barman-000003'] ... (+12 more)
  - Bring food: ['Bring food-Cook helper-000001', 'Bring food-Cook helper-000002', 'Bring food-Cook helper-000003'] ... (+12 more)
  - Deliver to customer: ['Deliver to customer-Waiter-000001', 'Deliver to customer-Waiter-000002', 'Deliver to customer-Waiter-000003'] ... (+22 more)
>>> Star